In [2]:
# Setup the Jupyter version of Dash
from jupyter_dash import JupyterDash

# Dash / visualization imports
import dash_leaflet as dl
from dash import dcc, html, dash_table
from dash.dependencies import Input, Output
import plotly.express as px
import pandas as pd
import base64

JupyterDash.infer_jupyter_proxy_config()

# Import your CRUD module
from CRUD_Python_Module import AnimalShelter


###########################
# Data / Database Setup
###########################

# Hard-coded AAC credentials
username = "aacuser"
password = "ChooseAStrongPassword123!"

# Connect to MongoDB through your CRUD class
shelter = AnimalShelter(username, password)

# Read all records for initial dashboard load
df = pd.DataFrame.from_records(shelter.read({}))

# Drop Mongo _id if present because Dash tables do not display ObjectId cleanly
if '_id' in df.columns:
    df.drop(columns=['_id'], inplace=True)


###########################
# Helper Functions
###########################

def get_rescue_query(rescue_type):
    """
    Return the MongoDB query for each rescue filter.
    These are based on the standard SNHU Grazioso Salvare rescue categories.
    """

    if rescue_type == "Water Rescue":
        return {
            "animal_type": "Dog",
            "breed": {
                "$in": [
                    "Labrador Retriever Mix",
                    "Chesapeake Bay Retriever",
                    "Newfoundland"
                ]
            },
            "sex_upon_outcome": "Intact Female",
            "age_upon_outcome_in_weeks": {"$gte": 26, "$lte": 156}
        }

    elif rescue_type == "Mountain or Wilderness Rescue":
        return {
            "animal_type": "Dog",
            "breed": {
                "$in": [
                    "German Shepherd",
                    "Alaskan Malamute",
                    "Old English Sheepdog",
                    "Siberian Husky",
                    "Rottweiler"
                ]
            },
            "sex_upon_outcome": "Intact Male",
            "age_upon_outcome_in_weeks": {"$gte": 26, "$lte": 156}
        }

    elif rescue_type == "Disaster or Individual Tracking":
        return {
            "animal_type": "Dog",
            "breed": {
                "$in": [
                    "Doberman Pinscher",
                    "German Shepherd",
                    "Golden Retriever",
                    "Bloodhound",
                    "Rottweiler"
                ]
            },
            "sex_upon_outcome": "Intact Male",
            "age_upon_outcome_in_weeks": {"$gte": 20, "$lte": 300}
        }

    # Reset / no filter
    return {}


def load_data(rescue_type):
    """
    Query MongoDB and return a DataFrame for the selected rescue type.
    """
    query = get_rescue_query(rescue_type)
    data = pd.DataFrame.from_records(shelter.read(query))

    if '_id' in data.columns:
        data.drop(columns=['_id'], inplace=True)

    return data


def build_pie_chart(dataframe):
    """
    Build second chart. Here we use a pie chart of breed counts.
    """
    if dataframe.empty or 'breed' not in dataframe.columns:
        return px.pie(
            names=["No Data"],
            values=[1],
            title="Breed Distribution"
        )

    breed_counts = dataframe['breed'].value_counts().reset_index()
    breed_counts.columns = ['breed', 'count']

    fig = px.pie(
        breed_counts,
        names='breed',
        values='count',
        title='Breed Distribution'
    )
    return fig


def build_map(dataframe):
    """
    Build map markers from the filtered data.
    Uses location_lat and location_long.
    """
    markers = []

    if dataframe.empty:
        return [dl.Marker(position=[30.75, -97.48], children=[dl.Tooltip("No results")])]

    for _, row in dataframe.iterrows():
        try:
            lat = float(row["location_lat"])
            lon = float(row["location_long"])
            name = row.get("name", "Unknown")
            breed = row.get("breed", "Unknown")
            animal_id = row.get("animal_id", "Unknown")

            markers.append(
                dl.Marker(
                    position=[lat, lon],
                    children=[
                        dl.Tooltip(f"{name} | {breed} | {animal_id}")
                    ]
                )
            )
        except:
            # Skip rows with bad/missing coordinates
            continue

    if not markers:
        markers = [dl.Marker(position=[30.75, -97.48], children=[dl.Tooltip("No valid map points")])]

    return markers


###########################
# Dashboard App
###########################

app = JupyterDash(__name__)

# Encode logo image
image_filename = 'Grazioso Salvare Logo.png'
encoded_image = base64.b64encode(open(image_filename, 'rb').read()).decode()

app.layout = html.Div([
    
    # Header / Logo / Name
    html.Center([
        html.Img(
            src='data:image/png;base64,{}'.format(encoded_image),
            style={'height': '250px'}
        ),
        html.H1("Grazioso Salvare Rescue Dashboard"),
        html.H3("Steven V. Burgett")
    ]),

    html.Hr(),

    # Rescue Filter Controls
    html.Div([
        html.Label("Select Rescue Type:", style={'font-weight': 'bold'}),
        dcc.RadioItems(
            id='filter-type',
            options=[
                {'label': ' All Animals (Reset)', 'value': 'Reset'},
                {'label': ' Water Rescue', 'value': 'Water Rescue'},
                {'label': ' Mountain or Wilderness Rescue', 'value': 'Mountain or Wilderness Rescue'},
                {'label': ' Disaster or Individual Tracking', 'value': 'Disaster or Individual Tracking'}
            ],
            value='Reset',
            labelStyle={'display': 'block', 'margin-right': '15px'}
        )
    ], style={'margin-bottom': '20px'}),

    html.Hr(),

    # Data Table
    dash_table.DataTable(
        id='datatable-id',
        columns=[{"name": i, "id": i} for i in df.columns],
        data=df.to_dict('records'),
        page_size=15,
        sort_action='native',
        filter_action='native',
        row_selectable='single',
        selected_rows=[0] if len(df) > 0 else [],
        style_table={'overflowX': 'auto'},
        style_cell={
            'textAlign': 'left',
            'minWidth': '120px',
            'width': '120px',
            'maxWidth': '180px',
            'whiteSpace': 'normal'
        },
        style_header={
            'backgroundColor': 'rgb(230, 230, 230)',
            'fontWeight': 'bold'
        }
    ),

    html.Br(),

    # Charts Row
    html.Div([
        # Map
        html.Div([
            html.H4("Animal Location Map"),
            dl.Map(
                id='map-id',
                style={'width': '100%', 'height': '500px'},
                center=[30.75, -97.48],
                zoom=10,
                children=[
                    dl.TileLayer(),
                    dl.LayerGroup(id='map-layer')
                ]
            )
        ], style={'width': '50%', 'display': 'inline-block', 'verticalAlign': 'top'}),

        # Pie chart
        html.Div([
            dcc.Graph(id='chart-id')
        ], style={'width': '50%', 'display': 'inline-block'})
    ])
])


###########################
# Callback: Update Data Table
###########################
@app.callback(
    Output('datatable-id', 'data'),
    Output('datatable-id', 'columns'),
    Input('filter-type', 'value')
)
def update_dashboard(rescue_type):
    if rescue_type == "Reset":
        dff = load_data("Reset")
    else:
        dff = load_data(rescue_type)

    columns = [{"name": i, "id": i} for i in dff.columns]
    return dff.to_dict('records'), columns


###########################
# Callback: Update Map
###########################
@app.callback(
    Output('map-layer', 'children'),
    Input('datatable-id', 'derived_virtual_data')
)
def update_map(viewData):
    if viewData is None:
        dff = df.head(25).copy()
    else:
        dff = pd.DataFrame(viewData).head(25)

    return build_map(dff)


###########################
# Callback: Update Pie Chart
###########################
@app.callback(
    Output('chart-id', 'figure'),
    Input('datatable-id', 'derived_virtual_data')
)
def update_chart(viewData):
    if viewData is None:
        dff = df.head(500).copy()
    else:
        dff = pd.DataFrame(viewData).head(500)

    return build_pie_chart(dff)


###########################
# Run App
###########################
app.run_server()

Dash app running on https://manualenjoy-luckypanama-3000.codio.io/proxy/8050/
